# agenteval - a LoRA-trained judge (free Colab GPU)

Most of agenteval's judges are either deterministic or a hosted LLM. This
notebook trains a **small, self-hosted judge** instead: a LoRA adapter on top
of a tiny encoder that scores whether an agent's output is *faithful* to the
input. It trains on a free Colab T4 in a couple of minutes and, crucially,
**loads CPU-side for inference** - so the resulting judge can run inside the
eval harness (or a Streamlit app) with no GPU and no per-call API cost.

Pipeline: base encoder -> LoRA fine-tune (GPU) -> save adapter -> load on CPU
-> plug in as an `agenteval` judge.

> Runtime: set **Runtime -> Change runtime type -> T4 GPU** before running.


## 1. Install


In [ ]:
!pip -q install "transformers>=4.40" "peft>=0.11" "datasets>=2.19" "accelerate>=0.30" evaluate scikit-learn


## 2. A tiny faithfulness dataset

Each example is `(input, output, label)` where label 1 = the output is a
faithful, on-topic answer and 0 = it is wrong or hallucinated. In a real
setup you would export these from agenteval traces (the dashboard's per-case
table); here we hand-write a small set so the notebook is self-contained.


In [ ]:
import random
random.seed(0)

# (support message, agent answer, faithful?)
raw = [
    ('I was charged twice for my subscription', 'This is a billing issue about a duplicate charge.', 1),
    ('I was charged twice for my subscription', 'Here is a recipe for banana bread.', 0),
    ('The app crashes when I open settings', 'A technical crash in the settings screen.', 1),
    ('The app crashes when I open settings', 'Your subscription will renew next month.', 0),
    ('How do I change my email address?', 'This is an account request to update the email on file.', 1),
    ('How do I change my email address?', 'The package was delivered yesterday.', 0),
    ('Where is my order, it has not arrived', 'A shipping question about a late delivery.', 1),
    ('Where is my order, it has not arrived', 'Please reset your password to continue.', 0),
    ('I want a refund for a double charge', 'A billing refund request for a duplicate payment.', 1),
    ('I want a refund for a double charge', 'Try reinstalling the mobile app.', 0),
    ('Reset link never arrives in my inbox', 'A technical issue with password-reset email delivery.', 1),
    ('Reset link never arrives in my inbox', 'Your invoice total is $42.', 0),
]
# a little augmentation so train/test both have signal
data = raw * 6
random.shuffle(data)
split = int(0.8 * len(data))
train_rows, eval_rows = data[:split], data[split:]
len(train_rows), len(eval_rows)


## 3. Tokenize

We feed the judge `INPUT ... [SEP] OUTPUT ...` and ask it to classify
faithful / unfaithful.


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

BASE = 'distilroberta-base'   # small, CPU-friendly at inference
tok = AutoTokenizer.from_pretrained(BASE)

def to_ds(rows):
    return Dataset.from_dict({
        'text': ['INPUT: %s' % r[0] for r in rows],
        'text_pair': ['OUTPUT: %s' % r[1] for r in rows],
        'label': [r[2] for r in rows],
    })

def tokenize(batch):
    return tok(batch['text'], batch['text_pair'], truncation=True, max_length=128)

train_ds = to_ds(train_rows).map(tokenize, batched=True)
eval_ds  = to_ds(eval_rows).map(tokenize, batched=True)


## 4. LoRA-wrap the model

We freeze the base encoder and train only small LoRA adapters on the
attention projections - a few hundred KB of trainable weights instead of the
full model.


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=2)

lora = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['query', 'value'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # ~0.3% of params


## 5. Train (GPU)


In [ ]:
import numpy as np, evaluate
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

acc = evaluate.load('accuracy')
def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return acc.compute(predictions=preds, references=p.label_ids)

args = TrainingArguments(
    output_dir='out', per_device_train_batch_size=8,
    num_train_epochs=8, learning_rate=2e-4, logging_steps=10,
    eval_strategy='epoch', report_to='none',
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics,
)
trainer.train()
trainer.evaluate()


## 6. Save just the adapter

The saved artifact is only the LoRA weights + config (a few hundred KB), not
the whole base model.


In [ ]:
ADAPTER_DIR = 'agenteval-lora-judge'
model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
!du -sh {ADAPTER_DIR}
!ls -la {ADAPTER_DIR}


## 7. Load CPU-side and score

This is the important part: the trained judge runs on CPU with no API. You
would download `agenteval-lora-judge/` and load it wherever the eval runs.


In [ ]:
from peft import AutoPeftModelForSequenceClassification

cpu_model = AutoPeftModelForSequenceClassification.from_pretrained(ADAPTER_DIR).to('cpu').eval()
cpu_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)

def faithfulness_score(inp, out):
    enc = cpu_tok('INPUT: %s' % inp, 'OUTPUT: %s' % out,
                  return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        prob = cpu_model(**enc).logits.softmax(-1)[0, 1].item()
    return round(prob, 3)

print('faithful  :', faithfulness_score('I was charged twice', 'A billing issue about a duplicate charge.'))
print('unfaithful:', faithfulness_score('I was charged twice', 'Here is a banana bread recipe.'))


## 8. Plug it into agenteval

The harness only needs a `score(case, output) -> Judgement`. Here is a judge
that wraps the CPU model - drop it in alongside the deterministic and hosted-
LLM judges, and it flows through the same scorecard and regression gate.


In [ ]:
# evalcore/judges_lora.py  (sketch)
from evalcore.judges import Judge, Judgement

class LoRAFaithfulnessJudge(Judge):
    name = 'lora_faithfulness'
    def __init__(self, adapter_dir, threshold=0.5):
        from peft import AutoPeftModelForSequenceClassification
        from transformers import AutoTokenizer
        self.model = AutoPeftModelForSequenceClassification.from_pretrained(adapter_dir).to('cpu').eval()
        self.tok = AutoTokenizer.from_pretrained(adapter_dir)
        self.threshold = threshold
    def score(self, case, output):
        import torch
        enc = self.tok('INPUT: %s' % case.input, 'OUTPUT: %s' % output,
                       return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            p = self.model(**enc).logits.softmax(-1)[0, 1].item()
        return Judgement(self.name, p, p >= self.threshold, 'lora faithfulness %.2f' % p)

# usage:  run_eval(target, cases, [LoRAFaithfulnessJudge('agenteval-lora-judge')])


---
Trains in minutes on a free T4, runs on CPU afterward, and slots into the same
`Judge` interface as everything else. Grow the dataset from real agenteval
traces to make the judge sharper.
